# Densest subgraph

## Basic Graph Class

We already provide the basic implementation of a graph class.

Iterating over nodes can be done by using ```for u in G.nodesToEdges``` and iterating over the edges incident upon node ```u``` can be done using ```for v in G.nodesToEdges[u]```. To check whether an edge $(u,v)$ exists use ```G.edgeExists(u,v)```.

Graphs can be read from files using the ```readFromFile```-procedure. Using the optional argument ```verticesToIgnore``` which expects a ```set()``` as input, you may exclude some nodes from the graph, i.e., if ```verticesToIgnore``` is some set $S$ then it will load the graph $G=(V\setminus S,E[V\setminus S])$.  Note that you might have to delete header-rows from some data files to make the procedure work.

In [33]:
class Graph:
    def __init__(self):
        self.numNodes = 0
        self.numEdges = 0

        self.edges = set()
        self.nodesToEdges = {}

    def addEdge(self,u,v):
        if u == v or v in self.nodesToEdges and u in self.nodesToEdges[v]:
            return
            
        self.numEdges += 1
        self.addNeighbor(u,v)
        self.addNeighbor(v,u)
        self.edges.add((u,v)) # we only add one pair (u,v) and not (v,u)

    def removeEdge(self,u,v):
        self.numEdges -= 1
        self.nodesToEdges[u].remove(v)
        self.nodesToEdges[v].remove(u)

        if (u,v) in self.edges:
            self.edges.remove((u,v))
        else: # (v,u) in self.edges
            self.edges.remove((v,u))

    def addNeighbor(self,u,v):
        if u not in self.nodesToEdges:
            self.numNodes += 1
            self.nodesToEdges[u] = set()

        self.nodesToEdges[u].add(v)

    def edgeExists(self, u, v):
        return ((u,v) in self.edges or (v,u) in self.edges)
        
    def degree(self, u):
        return len(self.nodesToEdges[u])

    def readFromFile(self, filePath, separator=',', verticesToIgnore=set()):
        with open(filePath, 'r') as f:
            for line in f:
                split = line.split(separator)
                u = int(split[0])
                v = int(split[1])

                if u in verticesToIgnore or v in verticesToIgnore:
                    continue
                    
                self.addEdge(u, v)

        print(f'Finished reading graph with {self.numNodes} nodes and {self.numEdges} edges.')

## Basic Linked List Data Structure

First, implement your own linked list data structure ```LinkedList``` in which each element is from the class ```LinkedListElement```.

In [34]:
class LinkedListElement:
    def __init__(self,id,prevElement=None,nextElement=None):
        self.id = id
        self.prevElement = prevElement
        self.nextElement = nextElement

class LinkedList:
    def __init__(self):
        self.head = None
        self.tail = None
        self.size = 0

    def appendElement(self, element: LinkedListElement):
        if self.head is None:
            self.head = element
            self.tail = element
        else:
            element.prevElement = self.tail
            self.tail.nextElement = element
            self.tail = element
        self.size += 1

    def removeElement(self, element: LinkedListElement):
        if element == self.head:
            self.head = element.nextElement

        if element == self.tail:
            self.tail = element.prevElement

        if element.prevElement:
            element.prevElement.nextElement = element.nextElement
        if element.nextElement:
            element.nextElement.prevElement = element.prevElement

        element.nextElement = None
        element.prevElement = None
        self.size -= 1
        
    def pop(self):
        if not self.tail:
            return None
        node_to_remove = self.tail
        self.removeElement(node_to_remove)
        return node_to_remove

## Implementation of the Greedy Peeling Algorithm

Next, implement the greedy peeling algorithm. The function should be callable using ```densestSubgraphGreedyPeeling(G)``` where ```G``` is a graph using the graph class from above. Besides G, the function may accept more optional arguments.

In [38]:
def densestSubgraphGreedyPeeling(G):
    max_id = max(G.nodesToEdges) # needed to allow non-continuous labeling
    d = [0] * (max_id + 1)
    # im assuming here that the vertices are named 0 - n <-- now also fine with a few "holes"
    for e in G.edges:
        d[e[0]] += 1
        d[e[1]] += 1

    L = list()
    for _ in range(0, G.numNodes):
        L.append(LinkedList())

    k_star = float('inf')
    pointers_to_vertices = [None] * (max_id + 1)  # needed to allow non-continuous labeling
    #for v, degree in enumerate(d):
    for v in G.nodesToEdges:   # works with a non-continuous degree list
        degree = d[v]
        element = LinkedListElement(v)
        L[degree].appendElement(element)
        pointers_to_vertices[v] = element

        if degree < k_star:
            k_star = degree

    #best_S = G.edges.copy() # O(m)
    best_num_edges = G.numEdges
    best_d_S = G.numEdges / G.numNodes
    removed_vertices = []
    best_removed_count = 0
    num_nodes = G.numNodes
    while num_nodes != 1:
        v_star = L[k_star].pop()
        removed_vertices.append(v_star.id)
        neighbors = G.nodesToEdges[v_star.id].copy() # O(m)
        for u in neighbors:
            G.removeEdge(v_star.id, u)
            current = pointers_to_vertices[u]
            L[d[current.id]].removeElement(current) # remove from L[d[u]]
            d[current.id] -= 1
            L[d[current.id]].appendElement(current) # add to L[d[u]-1]

            if d[current.id] < k_star:
                k_star = d[current.id]
        num_nodes -= 1


        d_S = G.numEdges / num_nodes
        if d_S > best_d_S:
            best_num_edges = G.numEdges
            best_d_S = d_S
            #best_S = G.edges.copy() # O(m) <- we are inside a O(n) loop, this would result in O(n*m)
            best_removed_count = len(removed_vertices)

        while k_star < len(L) and L[k_star].size <= 0:
            k_star = (k_star + 1) % G.numNodes

    best_S = set(removed_vertices[best_removed_count:]) # "get back" all vertices that were removed, after optimum was found
    best_S.add(L[k_star].head.id) # also add last vertex still in the list

    return best_S, best_num_edges, best_d_S

def runDisjointSubgraphExtraction(filePath):
    verticesToIgnore = set()
    results = []

    for round_idx in range(1, 6):
        working_graph = Graph()
        working_graph.readFromFile(filePath, ',', verticesToIgnore)
        vertex_set, _, density = densestSubgraphGreedyPeeling(working_graph)

        results.append({
            'round': round_idx,
            'vertex_set': set(vertex_set),
            'num_vertices': len(vertex_set),
            'density': density,
        })
        verticesToIgnore.update(vertex_set)

    return results

testGraph = Graph()
testGraph.addEdge(0,1)
testGraph.addEdge(1,2)
testGraph.addEdge(1,3)
testGraph.addEdge(1,25)
testGraph.addEdge(2,3)
testGraph.addEdge(2,25)
testGraph.addEdge(3,25)
densestSubgraphGreedyPeeling(testGraph)

({1, 2, 3, 25}, 1.5)

## Experiments for Densest Subgraph

Now, the experiments for the two datasets follow.

### Experiments for the OpenFlights Dataset

In [37]:
openflights_disjoint_subgraphs = runDisjointSubgraphExtraction('data/openflights_edges_cleaned.csv')

for result in openflights_disjoint_subgraphs:
    print(f"Round {result['round']}:")
    print(f"|S| = {result['num_vertices']}, density = {result['density']}\n")

openflights_disjoint_subgraphs

Finished reading graph with 3214 nodes and 18858 edges.
successfully calculated 1. dense subgraph, with:
|S| = 180 and density = 25.377777777777776

Finished reading graph with 2643 nodes and 7349 edges.
successfully calculated 2. dense subgraph, with:
|S| = 53 and density = 14.754716981132075

Finished reading graph with 2513 nodes and 5716 edges.
successfully calculated 3. dense subgraph, with:
|S| = 29 and density = 7.482758620689655

Finished reading graph with 2415 nodes and 4953 edges.
successfully calculated 4. dense subgraph, with:
|S| = 57 and density = 5.385964912280702

Finished reading graph with 2234 nodes and 4027 edges.
successfully calculated 5. dense subgraph, with:
|S| = 18 and density = 4.611111111111111



### Experiments for the Facebook Dataset

## Experiments for Optimal Quasi Clique

Now re-run the experiments from above with the new objective function and compare the results to the previous outputs.

### Experiments for the OpenFlights Dataset

### Experiments for the Facebook Dataset